### Transform Results Data

In [0]:
%run ../00-common/01-environment-config

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.results"
silver_table = f"{catalog_name}.{silver_schema}.results"

#### 1. Read results data

In [0]:
results_df = spark.read.table(bronze_table)
display(results_df)

#### 2. Drop Unncessesory columns

In [0]:
results_selected_df = results_df.drop("url")
display(results_selected_df)

#### 3. Standardize Columns

In [0]:
results_standardized_df = results_selected_df.withColumnsRenamed({
    "raceName": "race_name",
    "constructorId": "constructor_id",
    "driverId": "driver_id",
    "positionText": "finish_position_text",
})

### 3.1 Rename columns to make them more meaningful

In [0]:
results_renamed_df = results_standardized_df.withColumnsRenamed({
    "date": "race_date",
    "grid": "grid_position",
    "laps" : "completed_laps",
    "number" : "car_number",
    "position" : "finish_position",
})

In [0]:
results_count = results_renamed_df.count()
display(results_count)

#### 4. Bussiness Keys validation, filter out rows where these are Null

In [0]:
import pyspark.sql.functions as F
results_valid_df = results_renamed_df.filter(
    F.col("season").isNotNull() &
    F.col("round").isNotNull() &
    F.col("constructor_id").isNotNull() &
    F.col("driver_id").isNotNull()
)

display(results_valid_df)

In [0]:
display(results_df.count() - results_valid_df.count())

### 5 Remove duplicate rows 

In [0]:
null_counts_df = results_valid_df.select([
    F.sum(F.col(column).isNull().cast("int")).alias(column)
    for column in results_valid_df.columns
])

display(null_counts_df)

In [0]:
results_distinct_df = results_valid_df.dropDuplicates(['season', 'round', 'constructor_id', 'driver_id'])

In [0]:
display(results_valid_df.count() - results_distinct_df.count())

### 6. Transform Values of Columns race_name to Title Case

In [0]:
results_final_df = results_distinct_df.withColumns({
    "race_name": F.initcap(F.col("race_name")),
})

display(results_final_df)

### 7. Writing data to bronze table

In [0]:
(
    results_final_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(silver_table)
)

In [0]:
display(spark.table(silver_table))